<a href="https://colab.research.google.com/github/Santosh-S321/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Santosh-S321/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess, json
os.chdir("/content")
REPO_URL = "https://github.com/Santosh-S321/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

## 1. Ranked actions + reason codes

Rebuilding the grouped-split model from ML-08/09 (the honest version, not the
misleading random-split one), scoring every page, then assigning one reason code per
page based on which signals are actually extreme for it — so a reviewer sees not just
"review this" but "review this, specifically because X."

In [2]:
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "engagement_rate", "scroll_rate"]
features = [c for c in features if c in df.columns]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                             class_weight="balanced_subsample", random_state=42, n_jobs=-1)
rf.fit(X.iloc[train_idx], y.iloc[train_idx])

# Score the full dataset for the playbook (train on grouped split, score everyone)
df["model_score"] = rf.predict_proba(X)[:, 1]

def assign_reason(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page", "review_for_refresh"
    elif row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        return "low_ctr_visible_page", "review_for_ctr_fix"
    elif row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        return "thin_visible_page", "review_for_expansion"
    else:
        return "general_review", "monitor"

df[["reason_code", "action"]] = df.apply(assign_reason, axis=1, result_type="expand")

queue = df.sort_values("model_score", ascending=False)[
    ["content_id", "client_id", "model_score", "reason_code", "action",
     "impressions_90d", "avg_position", "days_since_last_update", "ctr", "word_count"]
]
print(queue.head(15))

p20 = precision_at_k(df.iloc[test_idx]["model_score"].values, y.iloc[test_idx].values, 20)
p50 = precision_at_k(df.iloc[test_idx]["model_score"].values, y.iloc[test_idx].values, 50)
base_rate = y.iloc[test_idx].mean()
print(f"\nGrouped-split held-out precision — P@20: {p20:.3f}, P@50: {p50:.3f}, base rate: {base_rate:.3f}")

                 content_id          client_id  model_score  \
15033  content_610fcc044e74  client_7f2253d7e2     0.936281   
23866  content_1e72ec381b2c  client_7f2253d7e2     0.935393   
1897   content_714cd092a56f  client_7f2253d7e2     0.934595   
8723   content_9e3689ef2df5  client_7f2253d7e2     0.932742   
6774   content_7a21f1aac586  client_7f2253d7e2     0.932453   
12577  content_82ea9307e1ab  client_7f2253d7e2     0.931518   
27422  content_69cfaabeda16  client_7f2253d7e2     0.931144   
3941   content_964dd0100c99  client_7f2253d7e2     0.930960   
24470  content_a0f750ca1b2a  client_7f2253d7e2     0.930767   
2866   content_a27e15f618c2  client_7f2253d7e2     0.930673   
23727  content_6eeb07cca243  client_7f2253d7e2     0.930373   
17879  content_341b1ecdea85  client_7f2253d7e2     0.929806   
13175  content_3585f0ab30e6  client_7f2253d7e2     0.927004   
18406  content_d75823fc91dc  client_7f2253d7e2     0.926431   
7672   content_0a08fba69bf3  client_7f2253d7e2     0.92

## 2. Intended use and limits

**Intended use:** a content reviewer with limited weekly capacity uses this ranked
queue to decide which pages to look at first. It ranks and flags candidates for
human review — it does not decide, publish, or fix anything on its own.

**Where it stops being valid:**
- The label (`trend_direction`-derived) is a current-window proxy, not an observed
  future outcome — the model ranks by association with this proxy, at Precision@20 =
  0.450 and Precision@50 = 0.600, against a base rate of 0.559. Note Precision@20 sits
  *below* the base rate — at the very top of the list, this model is not clearly
  outperforming random ranking. Precision@50 does beat the base rate modestly. These
  numbers describe this one portfolio slice, this one grouped split — not a universal
  claim about content refresh in general.
- Cross-sectional data: this cannot support "refreshing this page will cause traffic
  to recover" — only "pages with these characteristics are associated with the
  proxy-decline label." That's decision-support language, not causal language.
- If a client's pages were chosen non-randomly for review in the past (selection
  bias, same caveat as the Refresh ROI paper finding reviewed in ML-09), some of any
  observed lift on reviewed pages reflects the choosing, not the review itself.

In [3]:
print(f"Precision@20: {p20:.3f} | Precision@50: {p50:.3f} | Base rate: {base_rate:.3f}")

Precision@20: 0.450 | Precision@50: 0.600 | Base rate: 0.559


## 3. Human review + the no-go list

**Human review checklist before acting on any flagged page:**
- Confirm the traffic pattern isn't consolidation (a sibling page absorbing demand)
  or seasonality, per the decline-vs-look-alikes checklist from the lane guide.
- Check the client's actual history/context — a `days_since_last_update` value could
  be stale/unrecorded rather than genuinely untouched.
- Verify the reason code's underlying numbers still look real (not a data artifact).

**What should NEVER be automated:**
- Auto-publishing content changes based on the model's score alone.
- Auto-pruning or auto-merging pages without a human confirming consolidation logic.
- Treating a high score as proof a page is failing — per the paper's own Myth 2
  finding, flagged pages are often the *visible* pages worth improving, not failures.
- Any claim to a client that this system "predicts Google's algorithm" or
  "guarantees recovery" — banned language per writing-honest-claims.

In [4]:
print("No computation needed — this section is policy, not data.")

No computation needed — this section is policy, not data.


## 4. Monitoring / retrain triggers

**Signals that the playbook has gone stale:**
- Precision@50 on a fresh held-out month drops meaningfully below 0.600 — recheck
  features and label definition.
- The base rate (proportion of pages proxy-labeled "declining") shifts significantly
  from 0.559 — the underlying content mix or client portfolio has changed.
- Feature distributions drift (e.g., median `impressions_90d` shifts materially from
  731) — retrain rather than keep scoring on stale assumptions.

**Retrain cadence:** quarterly at minimum, or immediately if a monitoring trigger
above fires. Always re-validate with a grouped-by-client split, never a random split
(per ML-09's finding that random-split precision was misleadingly perfect — 1.00 @20
vs. this honest 0.450).

In [5]:
print(f"Reference values to monitor against — base rate: {base_rate:.3f}, "
      f"median impressions_90d: {df['impressions_90d'].median():.0f}")

Reference values to monitor against — base rate: 0.559, median impressions_90d: 731


## 5. Exports for the paper

Writing the ranked queue to `work/outputs/` (excluded from git by the CI leak-guard,
regenerated on every run) and the metrics JSON (committed — these are the receipts
the paper's numbers trace back to).

In [6]:
os.makedirs("work/outputs", exist_ok=True)

queue.to_csv("work/outputs/action_playbook_queue.csv", index=False)

metrics = {
    "precision_at_20": float(p20),
    "precision_at_50": float(p50),
    "base_rate": float(base_rate),
    "split_strategy": "client_holdout",
    "features_used": features,
    "n_rows_scored": int(len(df)),
}
with open("work/outputs/action_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Wrote:")
print(" - work/outputs/action_playbook_queue.csv")
print(" - work/outputs/action_playbook_metrics.json (commit this one)")
print(json.dumps(metrics, indent=2))

Wrote:
 - work/outputs/action_playbook_queue.csv
 - work/outputs/action_playbook_metrics.json (commit this one)
{
  "precision_at_20": 0.45,
  "precision_at_50": 0.6,
  "base_rate": 0.5594424958464095,
  "split_strategy": "client_holdout",
  "features_used": [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "engagement_rate",
    "scroll_rate"
  ],
  "n_rows_scored": 30000
}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.